In [124]:
import method as mth
import importlib, time
importlib.reload(mth)
from method import *

Generate Circuit

In [145]:
n, d, k = 24, 80, 4
print("qubit number is", n, ",depth is", d, ",T-gate number is", k)

#Generate random local circuit
circ = random_local_1d_clifford_plus_kT(n, d, k, seed=1238)

print(circ)
print("Total T gates:", sum(1 for op in circ.all_operations() if op.gate == cirq.T))

qubit number is 24 ,depth is 80 ,T-gate number is 4
0: ───────────X───S──────────H──────×───S──────────S──────────H──────────H────────────────────────────@──────────────H──────@──────────────H──────@───S─────────────────×───S──────────H──────@───S^-1───────S^-1───@───S──────────S^-1───@───S^-1──────────────×───H──────────S^-1───×───H──────────S──────×──────────────S──────X───S──────────S──────@───S─────────────────────H─────────────────────S──────────S^-1───×───S──────────S^-1───@──────────────S^-1───×───H──────────S──────@───S^-1──────────────────H──────────H──────@──────────────S^-1───X───S──────────S──────@───S^-1───────S──────×──────────────S──────@───S^-1───────S^-1───@───S^-1─────────────────────────────H──────────S^-1───────H──────@──────────────S^-1───@──────────────H──────X───H──────────S──────@──────────────S^-1───×───S^-1───────S──────────H──────────S^-1───@──────────────S──────X─────────────────────@───S^-1───────
              │                     │                       

# Entropy calculation by RT formula

In [146]:
start_time = time.perf_counter()


# obtain stabilizers 
stabs = stabilizers_through_clifford_plus_T(circ)
print(len(stabs))
qubits = list(circ.all_qubits())

# subregion A
A = qubits[:12]      

# logical operators supported on A
logics_A = logical_operators_supported_on_A(stabs, qubits, A)
print(len(logics_A))

# group logical operators into anti-commuting pairs and singlet
pairs_A, singlets_A = canonicalize_cirq_generators(logics_A, qubits)

# obtain bulk operator by logical_tomography
rho_bulk = logical_tomography(circ, pairs_A, singlets_A)

# stabilizer generators supported on A
stabs_A = stabilizers_supported_on_A(stabs,qubits,A)

# obtain area operator
area_operator = len(A)-len(stabs_A)-len(pairs_A)-len(singlets_A)

# entropy calculation
S_bulk = entropy(rho_bulk)
SA = area_operator + S_bulk

end_time = time.perf_counter()


print('area operator is ',area_operator, '   bulk entropy is ',S_bulk)
print('S(A) = ',SA)
print('time cost is ', end_time - start_time)


16
8
area operator is  7    bulk entropy is  4.355
S(A) =  11.355
time cost is  15.25597320892848


# Brute-force calculation of entropy

In [147]:
qubits = list(circ.all_qubits())
 
start_time = time.perf_counter()

rho_A = reduced_density_matrix_from_circuit(circ,A)
print('entropy S(A) is, ', entropy(rho_A))

end_time = time.perf_counter()

print('time cost is ', end_time - start_time)


entropy S(A) is,  11.354
time cost is  154.82268983288668
